In [18]:
# Core
import os
from dotenv import load_dotenv

In [19]:
# Libraries
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

In [20]:
# Setup Environment Vars
load_dotenv()
OPENAI_API_KEY = os.environ['OPENAI_API_KEY']
COHERE_API_KEY = os.environ['COHERE_API_KEY']

In [21]:
# Load OpenAI Models - Embeddings & Chats
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-3.5-turbo", max_tokens=300)

In [22]:
# Load Documents
pdf_path = 'rag_file_example.pdf'
loader = PyPDFLoader(pdf_path, extract_images=False)
pages = loader.load_and_split()

In [23]:
# Split by chunk
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,
    chunk_overlap=20,
    length_function=len,
    add_start_index=True,
)
chunks = text_splitter.split_documents(pages)

In [24]:
# Store chunks in Chroma
vectorstore = Chroma(
    embedding_function=embeddings,
    persist_directory="naive_db"
)

# Load Retriever
naive_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

In [25]:
# Reordenate documents
rerank = CohereRerank(model='rerank-english-v3.0', top_n=3)

compressor_retriever = ContextualCompressionRetriever(
    base_compressor=rerank,
    base_retriever=naive_retriever,
)

In [26]:
TEMPLATE = """
    You are a specialist in law and technology. Anwser tue question using the context provided.
    Query:
    {question}

    Context:
    {context}
"""

rag_prompt = ChatPromptTemplate.from_template(TEMPLATE)

In [27]:
# Setup retrival
setup_retrieval = RunnableParallel({'question': RunnablePassthrough(), 'context': compressor_retriever})

output_parser = StrOutputParser()

compressor_retrieval_chain = setup_retrieval | rag_prompt | llm | output_parser

In [28]:
compressor_retrieval_chain.invoke('What are the warning of the legal mark of AI?')

'Some potential warnings of the legal implications of AI include issues related to liability, accountability, privacy, data protection, bias and discrimination, intellectual property rights, and regulatory compliance. It is important for laws and regulations to keep pace with advancements in AI to address these concerns and ensure the responsible development and use of AI technologies.'